# Classes as Object Factories
This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Classes as Object Factories** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Classes as Object Factories**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Treat classes as runtime objects
- Understand instance creation clearly
- See how methods become bound to instances
- Connect attribute access to the object model


Creating an instance allocates a new object and then usually initializes it through `__init__`. The class object itself already exists and stores behavior that instances can use through attribute lookup.


In [1]:
class User:
    def __init__(self, name):
        self.name = name

u = User("Ada")
print(type(User))
print(type(u))
print(id(User), id(u))


<class 'type'>
<class '__main__.User'>
2451860996800 2451880472720


Disassembling instance creation shows that class calls are runtime operations like any other call. Python loads the class object, calls it, and receives a new instance back.


In [2]:
import dis

def make_user():
    class Temp:
        pass
    return Temp()

dis.dis(make_user)


  3           0 RESUME                   0

  4           2 PUSH_NULL
              4 LOAD_BUILD_CLASS
              6 LOAD_CONST               1 (<code object Temp at 0x0000023ADF7C02A0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_14492\727344181.py", line 4>)
              8 MAKE_FUNCTION            0
             10 LOAD_CONST               2 ('Temp')
             12 PRECALL                  2
             16 CALL                     2
             26 STORE_FAST               0 (Temp)

  6          28 PUSH_NULL
             30 LOAD_FAST                0 (Temp)
             32 PRECALL                  0
             36 CALL                     0
             46 RETURN_VALUE

Disassembly of <code object Temp at 0x0000023ADF7C02A0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_14492\727344181.py", line 4>:
  4           0 RESUME                   0
              2 LOAD_NAME                0 (__name__)
              4 STORE_NAME               1 (__module__)
              6 L

That is why they can be passed around, stored, and even customized dynamically.

That is usually stored in attributes like `self.name`.

When accessed through an instance, they become bound methods.

The object model is really a lookup and binding story.


One object carries data and exposes methods that use that data.


In [3]:
class User:
    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"Hello, {self.name}"

u = User("Ada")
print(u.name)
print(u.greet())


Ada
Hello, Ada


Writing `User("Ada")` is really calling the class object.


In [4]:
class Box:
    pass

print(callable(Box))
print(Box())


True


The instance is automatically supplied as the first argument when you call through the instance.


In [5]:
class Greeter:
    def hello(self, name):
        return f"Hello, {name}"

g = Greeter()
print(Greeter.hello)
print(g.hello)
print(g.hello("Python"))


<function Greeter.hello at 0x0000023ADF76B880>
<bound method Greeter.hello of <__main__.Greeter object at 0x0000023ADF7AFD10>>
Hello, Python


This is not only a classroom topic. **Classes as Object Factories** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Thinking `self` is a keyword rather than a conventional parameter name
- Using classes for everything even when a simpler structure would do
- Ignoring the fact that classes themselves are objects


- What is a class in Python?
- What happens when you instantiate a class?
- Why does a method receive `self`?


- Classes are callable objects.
- Instances hold state.
- Methods are bound through attribute access.


If this notebook did its job, **Classes as Object Factories** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
